# 03 · I file del device
*Blocco 2 · IoT Hub con ADR e DPS. Passo di lettura.*

Il notebook 02c ha registrato `robot-01`: ha scritto una cartella con i file della registrazione. Qui non si crea niente: si **guarda cosa c'è**, file per file, e a che serve.

**Prima:** il notebook 02c. **Dopo:** il notebook 04 controlla che l'Hub accetti il certificato.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

## La cartella del device
Ogni registrazione scrive una cartella con data e ora, `devices\<device>\<data-ora>`: prendiamo l'ultima.

In [ ]:
$id = 'robot-01'
$dir = (Get-ChildItem (Join-Path $cfg.Out "devices\$id") -Directory | Sort-Object Name | Select-Object -Last 1).FullName
"Cartella: $dir"
Get-ChildItem $dir | Select-Object Name, Length | Format-Table -AutoSize | Out-String

## Chi ha prodotto cosa
Tre fasi: presentarsi a DPS, chiedere il certificato, riceverlo. I **segreti** sono le chiavi private.

In [ ]:
$ruolo = @(
    [pscustomobject]@{ File = 'bootstrap-key.pem';   Fase = '1 · presentarsi a DPS';     Segreto = 'SI'; Cosa = 'chiave privata della credenziale di bootstrap' }
    [pscustomobject]@{ File = 'bootstrap-cert.pem';  Fase = '1 · presentarsi a DPS';     Segreto = 'no'; Cosa = 'certificato di bootstrap, firmato dalla CA della flotta' }
    [pscustomobject]@{ File = 'bootstrap-chain.pem'; Fase = '1 · presentarsi a DPS';     Segreto = 'no'; Cosa = 'certificato di bootstrap + CA: la catena mandata a DPS' }
    [pscustomobject]@{ File = 'device-key.pem';      Fase = '2 · chiedere il certificato'; Segreto = 'SI'; Cosa = 'chiave privata del device: nasce sul device, non ne esce' }
    [pscustomobject]@{ File = 'device.csr.pem';      Fase = '2 · chiedere il certificato'; Segreto = 'no'; Cosa = 'richiesta di certificato: chiave pubblica e nome del device' }
    [pscustomobject]@{ File = 'dps-registration.json'; Fase = '3 · riceverlo';           Segreto = 'no'; Cosa = 'risposta di DPS: stato, Hub assegnato, catena' }
    [pscustomobject]@{ File = 'issued-01.pem';       Fase = '3 · riceverlo';             Segreto = 'no'; Cosa = 'certificato del device, emesso da ADR' }
    [pscustomobject]@{ File = 'issued-02.pem';       Fase = '3 · riceverlo';             Segreto = 'no'; Cosa = 'certificato della Issuing CA' }
    [pscustomobject]@{ File = 'issued-03.pem';       Fase = '3 · riceverlo';             Segreto = 'no'; Cosa = 'certificato della Root CA' }
    [pscustomobject]@{ File = 'issued-chain.pem';    Fase = '3 · riceverlo';             Segreto = 'no'; Cosa = 'i tre insieme: con device-key.pem e il certificato client per l Hub' }
)
$ruolo | Where-Object { Test-Path (Join-Path $dir $_.File) } | Format-Table -AutoSize -Wrap | Out-String

## La credenziale di bootstrap
Serve solo a presentarsi a DPS. Dura 30 giorni, ed è firmata dalla CA di bootstrap della flotta, non da ADR.

In [ ]:
Get-CertSummary (Join-Path $dir 'bootstrap-cert.pem') | Format-List | Out-String

## Il certificato che ADR ha emesso
Tre certificati: il device, la Issuing CA che lo ha emesso, la Root CA che ha firmato la Issuing CA.

In [ ]:
foreach ($n in '01', '02', '03') { "--- issued-$n.pem"; Get-CertSummary (Join-Path $dir "issued-$n.pem") | Format-List | Out-String }

## Cosa abbiamo visto
- Per parlare con l'Hub servono **due file**: `issued-chain.pem` e `device-key.pem`.
- Le chiavi private sono due, `bootstrap-key.pem` e `device-key.pem`: la cartella non si pubblica.
- La chiave del device nasce sul device: nella richiesta (`device.csr.pem`) c'è solo la sua parte pubblica.

Prossimo passo: **04**, l'Hub accetta il certificato?